# 📘 Databricks Delta Live Tables (DLT) — Ultra Detailed (SQL + PySpark Side-by-Side)

---

## 🔷 1. What is Delta Live Tables (DLT)?

👉 DLT = **Declarative ETL Pipeline Framework**

👉 You define:
- Tables
- Transformations
- Data quality

👉 Databricks handles:
- Execution
- Scaling
- Dependencies (DAG)
- Monitoring

---

### 🧠 Core Idea

```
Define WHAT → DLT handles HOW
```

---

## 🔷 2. Table Types in DLT (Very Important 🔥)

---

## 🔹 1. LIVE TABLE (Materialized Table)

👉 Physical table stored in Delta

---

### SQL
```sql
CREATE LIVE TABLE customers
AS SELECT * FROM source;
```

### PySpark
```python
import dlt

@dlt.table
def customers():
    return spark.read.table("source")
```

---

## 🔹 2. STREAMING LIVE TABLE

👉 For streaming pipelines

---

### SQL
```sql
CREATE STREAMING LIVE TABLE customers_stream
AS SELECT * FROM STREAM(source);
```

### PySpark
```python
@dlt.table
def customers_stream():
    return spark.readStream.table("source")
```

---

## 🔹 3. TEMPORARY LIVE TABLE

👉 Intermediate (not persisted)

---

### SQL
```sql
CREATE TEMPORARY LIVE TABLE temp_data
AS SELECT * FROM source;
```

### PySpark
```python
@dlt.table(temporary=True)
def temp_data():
    return spark.read.table("source")
```

---

## 🔹 4. LIVE VIEW (Logical Layer 🔥)

👉 Not stored, used for transformations

---

### SQL
```sql
CREATE LIVE VIEW clean_view
AS SELECT * FROM source WHERE id IS NOT NULL;
```

### PySpark
```python
@dlt.view
def clean_view():
    return spark.read.table("source").filter("id IS NOT NULL")
```

---

## 🔹 5. MATERIALIZED VIEW (Physical Aggregation)

👉 Stored aggregated result

---

### SQL
```sql
CREATE LIVE TABLE mv_sales
AS SELECT country, COUNT(*) FROM source GROUP BY country;
```

### PySpark
```python
@dlt.table
def mv_sales():
    return spark.read.table("source").groupBy("country").count()
```

---

## 🔷 3. Medallion Architecture

---

| Layer | Purpose |
|------|--------|
| Bronze | Raw |
| Silver | Clean |
| Gold | Aggregated |

---

```
Bronze → Silver → Gold
```

---

## 🔷 4. End-to-End Pipeline

---

### SQL
```sql
-- Bronze
CREATE STREAMING LIVE TABLE bronze
AS SELECT * FROM cloud_files("/input", "json");

-- Silver
CREATE LIVE TABLE silver
AS SELECT * FROM bronze WHERE id IS NOT NULL;

-- Gold
CREATE LIVE TABLE gold
AS SELECT country, COUNT(*) FROM silver GROUP BY country;
```

---

### PySpark
```python
import dlt

@dlt.table
def bronze():
    return spark.readStream.format("cloudFiles")         .option("cloudFiles.format", "json")         .load("/input")

@dlt.table
def silver():
    return dlt.read("bronze").filter("id IS NOT NULL")

@dlt.table
def gold():
    return dlt.read("silver").groupBy("country").count()
```

---

## 🔷 5. Expectations (Data Quality 🔥)

---

### SQL
```sql
CREATE LIVE TABLE clean_data
CONSTRAINT valid_id EXPECT (id IS NOT NULL)
AS SELECT * FROM source;
```

### PySpark
```python
@dlt.table
@dlt.expect("valid_id", "id IS NOT NULL")
def clean_data():
    return spark.read.table("source")
```

---

### Types

| Function | Behavior |
|----------|---------|
| expect | Warn |
| expect_or_drop | Drop bad rows |
| expect_or_fail | Fail pipeline |

---

## 🔷 6. APPLY CHANGES INTO (CDC)

---

### SQL
```sql
APPLY CHANGES INTO target
FROM source
KEYS (id)
SEQUENCE BY timestamp
COLUMNS *
STORED AS SCD TYPE 1;
```

### PySpark
```python
from delta.tables import DeltaTable

delta = DeltaTable.forName(spark, "target")

delta.alias("t").merge(
    source.alias("s"),
    "t.id = s.id"
).whenMatchedUpdateAll()  .whenNotMatchedInsertAll()  .execute()
```

---

## 🔷 7. STREAM vs BATCH

---

### Streaming

#### SQL
```sql
SELECT * FROM STREAM(source);
```

#### PySpark
```python
spark.readStream.table("source")
```

---

### Batch

#### SQL
```sql
SELECT * FROM source;
```

#### PySpark
```python
spark.read.table("source")
```

---

## 🔷 8. Dependency Management (DAG)

---

```
bronze → silver → gold
```

👉 Automatically managed

---

## 🔷 9. Pipeline Modes

---

| Mode | Description |
|------|------------|
| Triggered | One-time run |
| Continuous | Continuous streaming |

---

## 🔷 10. Schema Evolution

---

### SQL
```sql
-- Handled automatically in DLT
```

### PySpark
```python
.option("mergeSchema", "true")
```

---

## 🔷 11. Monitoring & Lineage

---

DLT provides:
- DAG view
- Lineage tracking
- Logs
- Error tracking

---

## 🔷 12. Real-World Example

---

### PySpark
```python
@dlt.table
def bronze():
    return spark.readStream.format("cloudFiles")         .option("cloudFiles.format", "json")         .load("/input")

@dlt.table
def silver():
    return dlt.read("bronze").filter("amount > 0")

@dlt.table
def gold():
    return dlt.read("silver").groupBy("country").count()
```

---

## 🔷 13. Best Practices 🚀

- Use Medallion architecture
- Use expectations
- Use views for transformations
- Keep modular design

---

## 🔷 14. Common Mistakes 🚨

- ❌ Wrong table type
- ❌ No expectations
- ❌ Mixing batch/stream
- ❌ Ignoring dependencies

---

## 🔷 15. Interview Questions 🎯

👉 What is DLT?
→ Declarative ETL

👉 VIEW vs TABLE?
→ Logical vs Physical

👉 APPLY CHANGES?
→ CDC

👉 Expectations?
→ Data quality

---

## 🔷 🧠 Final Mental Model

```
Define Tables → Define Rules → DLT Builds DAG → Executes → Monitors
```

---

## 🔷 🚀 One-Line Summary

> DLT = Automated, scalable, declarative ETL pipelines with SQL + PySpark
